In [ ]:
import os

os.environ["AEQ_SHOW_PROGRESS"] = "FALSE"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from aequilibrae import Project
from aequilibrae.project.project import _upgrade
import sys
import numpy as np
from aequilibrae.paths import TrafficAssignment, TrafficClass
from aequilibrae.paths.vdf import bpr
import logging

import seaborn as sns
from scipy.stats import linregress

# pth = r"/mnt/d/models/Chicago"
pth = r"C:\Users\JakeMoss\models\Chicago"

In [ ]:
from aequilibrae.utils.logging_utils import basic_config

basic_config()

In [ ]:
# _upgrade(pth + r"/project_database.sqlite", pth + r"/public_transport.sqlite", None)

In [ ]:
proj = Project.from_path(pth)

net = proj.network
links = net.links.data
nodes = net.nodes.data

centroids = nodes.loc[nodes["is_centroid"] == 1, "node_id"].to_numpy()

In [ ]:
tolerance = 1e-5
max_iterations = 1000
num_threads = 16
method = "bfw"

## Matrices

In [ ]:
proj_matrices = proj.matrices
carDemand = proj_matrices.get_matrix("demand_omx")
carDemand.computational_view(["matrix"])

## Quetzal setup

In [ ]:
from quetzal.model import stepmodel

sm = stepmodel.StepModel()

In [ ]:
sm.zones = (
    nodes.loc[nodes["is_centroid"] == 1, ["node_id", "is_centroid", "geometry"]]
    .rename(columns={"node_id": "index", "is_centroid": "isCentroid"})
    .set_index("index")
)

In [ ]:
links["modes"] = [{"car"} for _ in range(len(links))]

In [ ]:
def flip_a_b_node(df):
    return df.assign(a_node=df["b_node"], b_node=df["a_node"])


columns = [
    "link_id",
    "a_node",
    "b_node",
    "direction",
    "link_type",
    "modes",
    "distance",
    "capacity_ab",
    "capacity_ba",
    "free_flow_time",
    "b",
    "power",
    "speed_ab",
    "speed_ba",
]

q_links = pd.concat(
    [
        links.loc[links["direction"] == 0, columns].assign(direction=1),
        flip_a_b_node(links.loc[links["direction"] == 0, columns]).assign(direction=-1),
        links.loc[links["direction"] == 1, columns],
        flip_a_b_node(links.loc[links["direction"] == -1, columns]),
    ],
    ignore_index=True,
)

for field, field_ab, field_ba in [
    ("capacity", "capacity_ab", "capacity_ba"),
    ("speed", "speed_ab", "speed_ba"),
]:
    q_links.loc[:, field] = float("NaN")
    for direction, col in [(1, field_ab), (-1, field_ba)]:
        filter = q_links["direction"] == direction
        if filter.any():
            q_links.loc[filter, field] = q_links.loc[filter, col]


q_links = q_links.loc[q_links.a_node != q_links.b_node]

q_links = q_links.rename(
    columns={
        "modes": "segments",
        "a_node": "a",
        "b_node": "b",
        "free_flow_time": "time",
        "b": "alpha",
        "power": "beta",
        "distance": "length",
    }
)

q_links["segments"] = q_links["segments"].apply(set)
q_links["vdf"] = "bpr"


q_links = q_links.set_index("link_id").drop(
    columns=["capacity_ab", "capacity_ba", "speed_ab", "speed_ba"]
)
q_links.index.name = "index"

centroid_connectors_slice = q_links["a"].isin(centroids) | q_links["b"].isin(centroids)

sm.road_links = q_links.loc[~centroid_connectors_slice].copy()

In [ ]:
sm.zone_to_road = q_links[centroid_connectors_slice]
sm.zone_to_road["direction"] = ""
sm.zone_to_road.loc[sm.zone_to_road["a"].isin(centroids), "direction"] = "access"
sm.zone_to_road.loc[sm.zone_to_road["b"].isin(centroids), "direction"] = "egress"
assert (sm.zone_to_road["direction"] != "").all()


In [ ]:
sm.road_links = sm.road_links.drop(columns=["direction"])

In [ ]:
sm.road_nodes = nodes.loc[
    nodes["is_centroid"] == 0, ["node_id", "is_centroid", "geometry"]
].set_index("node_id")

In [ ]:
sm.volumes = pd.DataFrame(
    {
        "origin": np.repeat(sm.zones.index, len(sm.zones)),
        "destination": np.tile(sm.zones.index, len(sm.zones)),
        "car": carDemand.matrix_view.flatten(),  #  PCE??
    }
)
sm.volumes = sm.volumes[
    (sm.volumes.drop(columns=["origin", "destination"]) != 0.0).any(axis=1)
].reset_index(drop=True)


In [ ]:
segments = ["car"]

vdf = {
    "bpr": "time * (1 + alpha * (flow / capacity) ** beta)",
}

In [ ]:
turn_penalties = pd.DataFrame([(1, 10293, 1, 0)], columns=["from_node", "via_node", "to_node", "penalty"])

In [ ]:
q_turn_penalties = {1: [1]}

In [ ]:
%%time
sm.step_road_pathfinder(
    method=method,
    maxiters=max_iterations,
    tolerance=tolerance * 100.0,  # quetzal's rgap is a percentage
    segments=segments,
    vdf=vdf,
    turn_penalties=q_turn_penalties,
    turn_penalty=0,
    num_cores=num_threads,
    return_car_los=False,
    log=True,
)

## Aequilibrae

In [ ]:
# sql = """
# --@ The *about* table holds information about the AequilibraE model
# --@ currently developed.
# --@
# --@ The **infoname** field holds the name of information being added
# --@
# --@ The **infovalue** field holds the information to add


# CREATE TABLE  if not exists about (infoname  TEXT UNIQUE NOT NULL,
#                                    infovalue TEXT
#                                   );
# --#
# INSERT INTO 'about' (infoname) VALUES('model_name');

# --#
# INSERT INTO 'about' (infoname) VALUES('region');

# --#
# INSERT INTO 'about' (infoname) VALUES('description');

# --#
# INSERT INTO 'about' (infoname) VALUES('author');

# --#
# INSERT INTO 'about' (infoname) VALUES('year');

# --#
# INSERT INTO 'about' (infoname) VALUES('scenario_description');

# --#
# INSERT INTO 'about' (infoname) VALUES('model_version');

# --#
# INSERT INTO 'about' (infoname) VALUES('project_id');

# --#
# INSERT INTO 'about' (infoname) VALUES('aequilibrae_version');

# --#
# INSERT INTO 'about' (infoname) VALUES('projection');

# --#
# INSERT INTO 'about' (infoname) VALUES('driving_side');

# --#
# INSERT INTO 'about' (infoname) VALUES('license');

# --#
# INSERT INTO 'about' (infoname) VALUES('scenario_name');

# --#
# INSERT INTO 'about' (infoname, infovalue) VALUES('allow_uturns', '0');
# """

# with proj.db_connection as conn:
#     conn.executescript(sql)


# # proj.upgrade()

In [ ]:
net.build_graphs()
graph = net.graphs["c"]

graph.set_graph("free_flow_time")
graph.set_blocked_centroid_flows(True)

In [ ]:
graph.set_turn_restrictions(pd.DataFrame([(1, 10293, 1, 0)], columns=["from_node", "via_node", "to_node", "penalty"]), allow_path_uturns=True)

In [ ]:
graph.has_turn_restrictions

In [ ]:
graph.allow_uturns_everywhere

In [ ]:
graph.allow_path_uturns

In [ ]:
from aequilibrae.paths.vdf import bpr

In [ ]:
assig = TrafficAssignment()

car_class = TrafficClass("car", graph, carDemand)
assig.add_class(car_class)

assig.set_vdf(bpr, {"alpha": "b", "beta": "power"})
assig.set_capacity_field("capacity")
assig.set_time_field("free_flow_time")
assig.set_algorithm(method)
assig.set_cores(num_threads, threading_threshold=10000, elementwise_cores=6)

assig.max_iter = max_iterations
assig.rgap_target = tolerance

In [ ]:
%%time
assig.execute()

In [ ]:
report = assig.report()

## Plots

In [ ]:
def plot_convergence(
    sm, aeq_report, name, max_iterations, method, tolerance, plot_time=True
):
    sns.set_theme(style="whitegrid", context="paper")

    aeq_results = aeq_report

    fig, ax = plt.subplots(figsize=(8, 6), dpi=150)
    palette = sns.color_palette()
    # palette = sns.color_palette("muted")

    if plot_time:
        x_label = "Time (s)"
        x_sm = sm.times
        x_aeq = aeq_results["time"]
    else:
        x_label = "Iterations"
        x_sm = range(len(sm.relgap))
        x_aeq = aeq_results.index

    sns.lineplot(
        x=x_sm,
        y=[x / 100.0 for x in sm.relgap],
        label="Quetzal relative gap",
        ax=ax,
        marker="X",
        markevery=int(max_iterations / 20),
        markersize=6,
        color=palette[0],
        linewidth=2,
    )
    sns.lineplot(
        x=x_aeq,
        y=aeq_results["rgap"],
        label="AequilibraE relative gap",
        ax=ax,
        marker="^",
        markevery=int(max_iterations / 20),
        markersize=7,
        color=palette[1],
        linewidth=2,
        # linestyle=":",
        # dash_capstyle="round",
    )

    ax.set_xlim(left=0)
    ax.set_yscale("log")
    ax.grid(True, which="minor", axis="y", linewidth=0.7, alpha=0.3)

    ax.set_xlabel(x_label, labelpad=10)
    ax.set_ylabel("Relative Gap", labelpad=10)
    ax.set_title(
        f"Convergence of Traffic Assignment for {name}\n{method.upper()}, Target rgap: {tolerance}",
        pad=14,
        fontweight="bold",
        fontsize=14,
    )
    ax.text(
        0.02,
        0.03,
        f"Markers shown every {int(max_iterations / 20)} iterations",
        transform=ax.transAxes,
        fontsize=9,
        color="black",
        bbox={
            "boxstyle": "round,pad=0.25",
            "facecolor": "white",
            "alpha": 0.9,
            "edgecolor": "0.9",
        },
    )
    legend = ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8")

    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    sns.despine(left=False, bottom=False)

    plt.tight_layout()
    plt.show()


plot_convergence(
    sm, report, "Chicago", max_iterations, method, tolerance, plot_time=True
)
# plot_convergence(sm, assig.report(), "Chicago", max_iterations, method, tolerance, plot_time=False)

In [ ]:
# Print convergence timing comparison
for target in [1e-4, 1e-5]:
    # Quetzal
    sm_gaps = [x / 100.0 for x in sm.relgap]
    sm_idx = next((i for i, g in enumerate(sm_gaps) if g <= target), None)
    sm_time = sm.times[sm_idx] if sm_idx is not None else None

    # AequilibraE
    aeq_idx = next((i for i, g in enumerate(report["rgap"]) if g <= target), None)
    aeq_time = report["time"].iloc[aeq_idx] if aeq_idx is not None else None

    print(f"--- Target rgap: {target:.0e} ---")
    if sm_time is not None:
        print(f"  Quetzal    : {sm_time:.2f}s (iter {sm_idx})")
    else:
        print(f"  Quetzal    : did not converge")
    if aeq_time is not None:
        print(f"  AequilibraE: {aeq_time:.2f}s (iter {aeq_idx})")
    else:
        print(f"  AequilibraE: did not converge")
    if sm_time is not None and aeq_time is not None:
        ratio = sm_time / aeq_time
        print(f"  Ratio (Quetzal/AequilibraE): {ratio:.2f}x")

In [ ]:
def plot_flow_comparison(
    sm,
    aeq_results,
    method,
):
    sns.set_theme(style="whitegrid", context="paper")

    fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

    quetzal_results = sm.road_links["flow"]

    x_data = quetzal_results
    y_data = aeq_results.loc[quetzal_results.index, "PCE_tot"]

    ax.scatter(x_data, y_data, alpha=0.5, s=3, label="Link flows")

    limit = max(*ax.get_xlim(), *ax.get_ylim())

    ax.set_xlim(left=0)
    ax.set_ylim(bottom=0)

    regression = linregress(x_data, y_data)
    x_line = np.array([0, limit])
    y_line = regression.intercept + regression.slope * x_line
    ax.plot(
        x_line,
        y_line,
        linestyle="--",
        linewidth=2,
        label=f"Regression  R²={regression.rvalue**2:.4f}\ny = {regression.slope:.4f}x + {regression.intercept:.4f}",
        color="red",
    )

    ax.plot(
        [0, limit],
        [0, limit],
        linestyle="-",
        color="grey",
        alpha=0.5,
        label="1:1 Line",
    )

    ax.set_xlabel("Quetzal Flow", labelpad=10)
    ax.set_ylabel("AequilibraE Flow", labelpad=10)
    ax.set_title(
        f"Quetzal vs AequilibraE Link Flows\n{method.upper()}, Relative gap < {tolerance}",
        pad=14,
        fontweight="bold",
        fontsize=14,
    )

    ax.legend(frameon=True, framealpha=0.9, edgecolor="0.8", loc="upper left")
    sns.despine(left=False, bottom=False)

    ax.set_aspect("equal", adjustable="box")

    plt.tight_layout()
    plt.show()


plot_flow_comparison(sm, assig.results(), method)

In [ ]:
quetzal_results = sm.road_links["flow"]
aeq_results = assig.results().loc[quetzal_results.index, "PCE_tot"]

In [ ]:
(quetzal_results - aeq_results).describe()

In [ ]:
aeq_results_100 = assig.results().copy()
quetzal_results_100 = sm.road_links["flow"].copy()

In [ ]:
(
    quetzal_results_100 - aeq_results_100.loc[quetzal_results_100.index, "PCE_tot"]
).describe()

In [ ]:
(assig.results()["PCE_tot"] - aeq_results_100["PCE_tot"]).describe()

In [ ]:
(quetzal_results - quetzal_results_100).describe()